# ASAP8 single-neuron change / omission response screen

This notebook is designed to answer the narrow cell-level question: **how many voltage-imaged VIP neurons show change responses, how many show omission responses, and what do the membrane-potential and spike data look like for those cells trial-by-trial?**

Principles:
- keep the primary readouts in native ASAP8 dF/F and detected spike times;
- use the same event indexing and nearby same-image control logic as the existing DoC notebook;
- classify responses at the **session × neuron** level first, then summarize unique tracked cells separately;
- show effect magnitude, trial consistency, confidence intervals, and statistical evidence rather than selecting cells from mean traces alone;
- do **not** temporally filter or smooth dF/F traces in either analysis or plotting.

Primary definitions used here:
- **Change signal:** change-window dF/F (0–0.25 s) minus nearby ordinary presentations of the same changed-to image.
- **Omission signal:** omission-cycle dF/F (0–0.50 s) minus nearby ordinary presentations of the expected image.
- **Image identity signal:** fraction of trial-wise image-response variance explained by image identity, assessed against label permutations.
- The same event/control comparisons are repeated using spike counts from the continuous ASAP8 trace.


In [ ]:
%load_ext autoreload
%autoreload 2

import os, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import wilcoxon
from matplotlib.lines import Line2D
from IPython.display import display, HTML

from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.voltage.dataset import DEPTH_GROUP_ORDER, build_voltage_session_table, build_voltage_roi_table
from vip_slap2_analysis.behavior.change_detection import build_change_detection_events
from vip_slap2_analysis.voltage.responses import build_single_trial_index
from vip_slap2_analysis.voltage.spikes import DETECTOR_VERSION, extract_session_spikes

assert DETECTOR_VERSION == "template_v1"
sns.set_style("white")
plt.rcParams.update({"legend.fontsize":10,"axes.labelsize":13,"axes.titlesize":14,"xtick.labelsize":11,"ytick.labelsize":11})
display(HTML("<style>.container { width:100% !important; }</style>"))


In [ ]:
# ------------------------------ Configuration ------------------------------
BASE_PATH = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics")
TARGET_MICE = [852835, 863774]
PARADIGMS = ["change_detection_passive"]
EXCLUDE_SESSION_TYPES = ["expression_check", "volume_imaging"]
SESSION_ORDER = ["A0","A1","A2","B0","B1","B2"]
TRACE_VARIANT = "dff_robust_f0_trial"
REGISTRATION_FILENAME = "roi_identity_registration.csv"
EXCLUDE_INVALID_ROIS = True
EXPECTED_F0_SMOOTH_SEC = 60.0

IMAGE_BASELINE_S = (-0.25, 0.0)
IMAGE_RESPONSE_S = (0.0, 0.25)
CHANGE_RESPONSE_S = (0.0, 0.25)
OMISSION_RESPONSE_S = (0.0, 0.50)
PREVIOUS_IMAGE_S = (-0.75, -0.50)
N_MATCHED_CONTROLS = 5
MATCH_POSITION_TOLERANCE = 2
MIN_EVENT_TRIALS = 8
MIN_TRIALS_PER_IMAGE = 5
MIN_IMAGES_FOR_IDENTITY = 3
N_BOOT = 2000
N_PERM_IDENTITY = 1000
FDR_ALPHA = 0.05
RNG_SEED = 17

SPIKE_KWARGS = dict(height_sigma=3.5, template_sigma=4.5, prominence_sigma=0.5)
RECOMPUTE_SPIKES = False
CACHE_DIR = Path.cwd() / "asap8_cell_response_cache"
SAVE_PATH = Path.cwd() / "asap8_cell_response_figures"
CACHE_DIR.mkdir(parents=True, exist_ok=True); SAVE_PATH.mkdir(parents=True, exist_ok=True)
SPIKE_CACHE = CACHE_DIR / "asap8_change_omission_spikes.pkl"

DEPTH_COLORS = {"<100 µm":"#EBA287","100–150 µm":"#d1e2b0",">150 µm":"#7bbcd5"}
SIGNAL_COLORS = {"image_identity":"#6f6f6f", "change":"#d95f02", "omission":"#1b9e77"}

def depth_group_from_um(depth):
    depth=float(depth)
    return "<100 µm" if depth < 100 else ("100–150 µm" if depth <= 150 else ">150 µm")


## Dataset, ROI registry, events, and continuous spike traces


In [ ]:
registry = VIPSessionRegistry.from_basepath(BASE_PATH)
sessions = build_voltage_session_table(
    registry, subject_ids=TARGET_MICE, paradigms=PARADIGMS,
    exclude_session_types=EXCLUDE_SESSION_TYPES, trace_variant=TRACE_VARIANT,
    expected_f0_smooth_sec=EXPECTED_F0_SMOOTH_SEC,
)
sessions = sessions[sessions["session_label"].astype(str).isin(SESSION_ORDER)].copy()
sessions["session_id"] = sessions["session_id"].astype(str)

rois = build_voltage_roi_table(
    sessions, registration_filename=REGISTRATION_FILENAME,
    exclude_invalid_rois=EXCLUDE_INVALID_ROIS,
)
rois["session_id"] = rois["session_id"].astype(str)
rois["depth_um"] = pd.to_numeric(rois["depth_um"], errors="coerce")
rois["depth_group"] = rois["depth_um"].map(depth_group_from_um)

events = build_change_detection_events(sessions)
events["session_id"] = events["session_id"].astype(str)
trial_index = build_single_trial_index(sessions, events)
trial_index["session_id"] = trial_index["session_id"].astype(str)

# Resolve the full-session dF/F H5 used by the established spike detector.
raw = registry.sessions(
    subject_ids=TARGET_MICE, paradigms=PARADIGMS,
    exclude_session_types=EXCLUDE_SESSION_TYPES,
).copy()
raw["session_id"] = raw["session_id"].astype(str)
trace_rows=[]
for _, row in raw.iterrows():
    if str(row["session_id"]) not in set(sessions["session_id"]):
        continue
    asset = registry.resolve_assets(row)
    trace_h5 = Path(asset.derived_dir) / "voltage" / f"voltage_session_traces_{TRACE_VARIANT}.h5"
    trace_rows.append(dict(session_id=str(asset.session_id), trace_h5=trace_h5))
trace_paths = pd.DataFrame(trace_rows).drop_duplicates("session_id")
sessions = sessions.merge(trace_paths, on="session_id", how="left")
sessions["spike_trace_available"] = sessions["trace_h5_y"].map(lambda p: isinstance(p, Path) and p.exists())

print(f"{len(sessions)} sessions · {int(rois['included'].sum())} included ROI observations")
display(sessions[["subject_id","session_id","session_label","session_order","dmd1_depth_um","dmd2_depth_um","spike_trace_available"]])


In [ ]:
# ------------------------------ Shared helpers ------------------------------
def finish_axis(ax):
    sns.despine(ax=ax); ax.tick_params(axis="both", labelsize=10)
    for spine in ax.spines.values(): spine.set_linewidth(1.5)

def decode_strings(values):
    return np.asarray([x.decode() if isinstance(x,(bytes,np.bytes_)) else str(x) for x in np.asarray(values).reshape(-1)])

def h5_roi_axis(group,dmd,source_roi):
    ids=decode_strings(group["roi_ids"][:]); label=f"DMD{int(dmd)}_ROI{int(source_roi)}"; hits=np.flatnonzero(ids==label)
    if not len(hits):
        parsed=np.array([int(re.findall(r"\d+",x)[-1]) for x in ids],int); hits=np.flatnonzero(parsed==int(source_roi))
    if not len(hits): raise KeyError(f"{label} is absent from H5 kept ROI axis")
    return int(hits[0])

def reconcile_timebase(t,n):
    t=np.asarray(t,float).reshape(-1)
    if len(t)==n: return t
    if len(t)<2: raise ValueError(f"Cannot reconcile {len(t)} time samples to {n} trace samples")
    dt=float(np.nanmedian(np.diff(t))); zero=min(int(np.nanargmin(np.abs(t))),n-1)
    return (np.arange(n,dtype=float)-zero)*dt

def window_slice(t,window):
    t=np.asarray(t,float); a,b=map(float,window)
    i0=int(np.searchsorted(t,a,side="left")); i1=int(np.searchsorted(t,b,side="left"))
    if i1<=i0: raise ValueError(f"Window {window} has no samples in [{t[0]}, {t[-1]}]")
    return slice(i0,i1)

def window_mean(y,t,window,axis=-1):
    return np.nanmean(np.asarray(y,float)[...,window_slice(t,window)],axis=axis)

def shifted_window_means(traces,t,offsets,window):
    out=np.full(len(traces),np.nan,float)
    for i,offset in enumerate(np.asarray(offsets,float)):
        if np.isfinite(offset): out[i]=window_mean(traces[i],t,(window[0]+offset,window[1]+offset))
    return out

def align_trials_to_true_onset(traces,t,offsets):
    traces=np.asarray(traces,float); aligned=np.full_like(traces,np.nan,dtype=float)
    for i,(trace,offset) in enumerate(zip(traces,np.asarray(offsets,float))):
        if np.isfinite(offset): aligned[i]=np.interp(t+offset,t,trace,left=np.nan,right=np.nan)
    return aligned

def indexed_event_table(session_id,dmd,event_type):
    sid=str(session_id)
    idx=trial_index[
        trial_index["session_id"].astype(str).eq(sid)
        & trial_index["dmd"].astype(int).eq(int(dmd))
        & trial_index["event_type"].astype(str).eq(str(event_type))
    ].copy()
    if "matched" in idx.columns: idx=idx[idx["matched"].astype(bool)]
    idx=idx.rename(columns={"onset_sec":"stored_onset_sec","image_name":"stored_image_name"})
    ev=events[events["session_id"].astype(str).eq(sid)].copy()
    needed=["session_id","event_id","onset_sec","image_name","image_label","is_change","is_omission","sequence_position_expected","next_event_id"]
    for col in needed:
        if col not in ev: ev[col]=np.nan
    ev=ev[needed].rename(columns={"onset_sec":"cycle_onset_sec","image_name":"event_image_name","image_label":"event_image_label"})
    out=idx.merge(ev,on=["session_id","event_id"],how="left",validate="many_to_one")
    return out.sort_values("trial_index").reset_index(drop=True)

def nearest_control_rows(label,onset_sec,sequence_position,controls,n_controls=N_MATCHED_CONTROLS,position_tolerance=MATCH_POSITION_TOLERANCE):
    if controls.empty or pd.isna(label): return controls.iloc[0:0]
    candidates=controls[controls["event_image_label"].astype(str).eq(str(label))].copy()
    pos0=pd.to_numeric(sequence_position,errors="coerce")
    if np.isfinite(pos0) and len(candidates):
        pos=pd.to_numeric(candidates["sequence_position_expected"],errors="coerce")
        near=np.abs(pos-float(pos0))<=float(position_tolerance)
        if near.any(): candidates=candidates[near]
    if not len(candidates): return candidates
    candidates["match_distance_sec"]=np.abs(pd.to_numeric(candidates["cycle_onset_sec"],errors="coerce")-float(onset_sec))
    return candidates.nsmallest(int(n_controls),"match_distance_sec")

def bh_fdr(p):
    p=np.asarray(p,float); out=np.full(len(p),np.nan,float); good=np.isfinite(p)
    vals=p[good]; m=len(vals)
    if not m: return out
    order=np.argsort(vals); ranked=vals[order]*m/np.arange(1,m+1)
    ranked=np.minimum.accumulate(ranked[::-1])[::-1]; ranked=np.clip(ranked,0,1)
    tmp=np.empty(m,float); tmp[order]=ranked; out[good]=tmp
    return out

def bootstrap_mean_ci(x,n_boot=N_BOOT,seed=RNG_SEED):
    x=np.asarray(x,float); x=x[np.isfinite(x)]
    if len(x)<2: return (np.nan,np.nan)
    rng=np.random.default_rng(seed); means=np.empty(int(n_boot),float)
    for i in range(int(n_boot)): means[i]=np.mean(rng.choice(x,size=len(x),replace=True))
    return tuple(np.quantile(means,[.025,.975]))

def signed_rank_p(x):
    x=np.asarray(x,float); x=x[np.isfinite(x)]
    if len(x)<2 or np.allclose(x,0): return 1.0
    try: return float(wilcoxon(x,zero_method="wilcox",alternative="two-sided").pvalue)
    except ValueError: return 1.0

def identity_fve(labels,values):
    labels=np.asarray(labels).astype(str); values=np.asarray(values,float); good=np.isfinite(values)
    labels,values=labels[good],values[good]
    if len(values)<2 or len(np.unique(labels))<2: return np.nan
    grand=float(np.mean(values)); total=float(np.sum((values-grand)**2))
    if total<=0: return np.nan
    stats=pd.DataFrame({"label":labels,"value":values}).groupby("label",observed=True)["value"].agg(["mean","size"])
    between=float(np.sum(stats["size"]*(stats["mean"]-grand)**2))
    return between/total

def identity_permutation_p(labels,values,n_perm=N_PERM_IDENTITY,seed=RNG_SEED):
    labels=np.asarray(labels).astype(str); values=np.asarray(values,float); good=np.isfinite(values)
    labels,values=labels[good],values[good]; obs=identity_fve(labels,values)
    if not np.isfinite(obs): return np.nan
    rng=np.random.default_rng(seed); null=np.empty(int(n_perm),float)
    for i in range(int(n_perm)): null[i]=identity_fve(rng.permutation(labels),values)
    return float((1+np.sum(null>=obs))/(len(null)+1))

def spike_counts(spike_times,onsets,window):
    spike_times=np.sort(np.asarray(spike_times,float)); onsets=np.asarray(onsets,float)
    a,b=map(float,window); left=np.searchsorted(spike_times,onsets+a,side="left"); right=np.searchsorted(spike_times,onsets+b,side="left")
    return right-left

def cell_meta(r):
    return dict(subject_id=str(r.subject_id),session_id=str(r.session_id),session_label=str(r.session_label),session_order=int(r.session_order),
                dmd=int(r.dmd),roi=int(r.roi),cell_id=str(r.cell_id),global_cell_id=str(getattr(r,"global_cell_id","")),
                manually_registered=bool(getattr(r,"manually_registered",False)),depth_um=float(r.depth_um),depth_group=str(r.depth_group))


In [ ]:
# Build / load detected spikes. The detector sees the unfiltered continuous dF/F trace.
if SPIKE_CACHE.exists() and not RECOMPUTE_SPIKES:
    spikes=pd.read_pickle(SPIKE_CACHE)
    print(f"Loaded {len(spikes):,} cached spikes from {SPIKE_CACHE}")
else:
    tables=[]
    for session in sessions.itertuples(index=False):
        if not bool(session.spike_trace_available):
            warnings.warn(f"{session.session_id}: continuous trace H5 missing; spike analyses skipped")
            continue
        session_rois=rois[rois["included"].astype(bool)&rois["session_id"].astype(str).eq(str(session.session_id))]
        roi_lookup={int(dmd):g["roi"].astype(int).tolist() for dmd,g in session_rois.groupby("dmd")}
        print(f"Detecting spikes: {session.session_label} · {session.session_id}")
        table=extract_session_spikes(session.trace_h5,rois=roi_lookup,**SPIKE_KWARGS)
        table.insert(0,"session_id",str(session.session_id)); tables.append(table)
    spikes=pd.concat(tables,ignore_index=True) if tables else pd.DataFrame()
    if len(spikes):
        spikes=spikes.merge(rois[["subject_id","session_id","session_label","session_order","dmd","roi","cell_id","global_cell_id","manually_registered","depth_um","depth_group"]],on=["session_id","dmd","roi"],how="left")
    spikes.to_pickle(SPIKE_CACHE)
    print(f"Saved {len(spikes):,} spikes to {SPIKE_CACHE}")

display(spikes.head())


In [ ]:
# Clock-overlap QC: event times should lie within the continuous trace timebase used for spikes.
qc=[]
for s in sessions.itertuples(index=False):
    if not bool(s.spike_trace_available): continue
    ev=events[events["session_id"].astype(str).eq(str(s.session_id))]
    with h5py.File(s.trace_h5,"r") as h5:
        tmins=[]; tmaxs=[]
        for key in [k for k in h5.keys() if str(k).startswith("DMD")]:
            t=np.asarray(h5[key]["timebase_sec"][:],float); tmins.append(np.nanmin(t)); tmaxs.append(np.nanmax(t))
    qc.append(dict(session_id=str(s.session_id),session_label=str(s.session_label),trace_start=min(tmins),trace_stop=max(tmaxs),
                   event_start=pd.to_numeric(ev["onset_sec"],errors="coerce").min(),event_stop=pd.to_numeric(ev["onset_sec"],errors="coerce").max()))
clock_qc=pd.DataFrame(qc); clock_qc["events_inside_trace"]=(clock_qc["event_start"]>=clock_qc["trace_start"]-1)&(clock_qc["event_stop"]<=clock_qc["trace_stop"]+1)
display(clock_qc)
if len(clock_qc) and not clock_qc["events_inside_trace"].all():
    warnings.warn("At least one session has event times outside the continuous trace timebase. Inspect clock_qc before interpreting spike-aligned results.")


## Trial-level dF/F tables and matched controls


In [ ]:
_IMAGE_CONTROL_CACHE={}

def load_image_control_table(h5,session_id,dmd,source_roi,axis):
    key=(str(session_id),int(dmd),int(source_roi))
    if key in _IMAGE_CONTROL_CACHE: return _IMAGE_CONTROL_CACHE[key]
    idx=indexed_event_table(session_id,dmd,"image")
    idx=idx[(~idx["is_change"].fillna(False).astype(bool)) & (~idx["is_omission"].fillna(False).astype(bool))].copy()
    stored_t=np.asarray(h5["timebase_sec/image"][:],float); parts=[]
    for path,q in idx.groupby("dataset_path",sort=False):
        q=q.sort_values("trial_index").copy(); rows=q["trial_index"].astype(int).to_numpy(); ds=h5[str(path)]
        t=reconcile_timebase(stored_t,int(ds.shape[-1])); arr=np.asarray(ds[rows,int(axis),:],float)
        q["image_baseline_dff"]=window_mean(arr,t,IMAGE_BASELINE_S)
        q["image_dff"]=window_mean(arr,t,IMAGE_RESPONSE_S)
        q["image_delta_dff"]=q["image_dff"]-q["image_baseline_dff"]
        q["cycle_dff"]=window_mean(arr,t,OMISSION_RESPONSE_S)
        parts.append(q)
    out=pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()
    _IMAGE_CONTROL_CACHE[key]=out
    return out

image_trial_rows=[]; image_curve_rows=[]; event_dff_rows=[]; event_curve_rows=[]
for session in sessions.itertuples(index=False):
    sid=str(session.session_id); session_rois=rois[rois["included"].astype(bool)&rois["session_id"].astype(str).eq(sid)]
    with h5py.File(session.single_trial_h5,"r") as h5:
        for r in session_rois.itertuples(index=False):
            dmd,roi=int(r.dmd),int(r.roi); g=h5[f"DMD{dmd}"]; axis=h5_roi_axis(g,dmd,roi); meta=cell_meta(r)
            controls=load_image_control_table(h5,sid,dmd,roi,axis)
            if len(controls):
                q=controls.copy(); q["image_label"]=q["event_image_label"].astype(str)
                image_trial_rows.append(pd.DataFrame({**{k:[v]*len(q) for k,v in meta.items()},"event_id":q["event_id"].to_numpy(),"onset_sec":pd.to_numeric(q["cycle_onset_sec"],errors="coerce").to_numpy(),"image_label":q["image_label"].to_numpy(),"image_delta_dff":q["image_delta_dff"].to_numpy(float)}))
                # Mean ordinary-image membrane potential, without temporal smoothing.
                stored_t=np.asarray(h5["timebase_sec/image"][:],float); pieces=[]
                for path,h in q.groupby("dataset_path",sort=False):
                    rows=h["trial_index"].astype(int).to_numpy(); ds=h5[str(path)]; t=reconcile_timebase(stored_t,int(ds.shape[-1])); pieces.append(np.asarray(ds[rows,axis,:],float))
                if pieces:
                    arr=np.concatenate(pieces,axis=0); image_curve_rows.append({**meta,"time":t,"mean_dff":np.nanmean(arr,axis=0),"n_trials":len(arr)})

            for event_type,window in [("change",CHANGE_RESPONSE_S),("omission",OMISSION_RESPONSE_S)]:
                sub=g[event_type]; traces=np.asarray(sub["traces"][:,axis,:],float); stored_t=np.asarray(h5[f"timebase_sec/{event_type}"][:],float); t=reconcile_timebase(stored_t,traces.shape[-1])
                idx=indexed_event_table(sid,dmd,event_type)
                if len(idx)!=len(traces) or not np.array_equal(idx["trial_index"].astype(int).to_numpy(),np.arange(len(traces))):
                    raise ValueError(f"{event_type} trial index mismatch for {sid} DMD{dmd}")
                offsets=pd.to_numeric(idx["cycle_onset_sec"],errors="coerce").to_numpy()-pd.to_numeric(idx["stored_onset_sec"],errors="coerce").to_numpy()
                response=shifted_window_means(traces,t,offsets,window); previous=shifted_window_means(traces,t,offsets,PREVIOUS_IMAGE_S)
                aligned=align_trials_to_true_onset(traces,t,offsets)
                for i,row in enumerate(idx.itertuples(index=False)):
                    matches=nearest_control_rows(row.event_image_label,row.cycle_onset_sec,row.sequence_position_expected,controls)
                    control_col="image_dff" if event_type=="change" else "cycle_dff"
                    matched_mean=float(np.nanmean(pd.to_numeric(matches[control_col],errors="coerce"))) if len(matches) else np.nan
                    control_onsets=pd.to_numeric(matches["cycle_onset_sec"],errors="coerce").dropna().to_numpy(float) if len(matches) else np.array([],float)
                    event_dff_rows.append({**meta,"event_type":event_type,"trial_index":int(i),"event_id":row.event_id,"onset_sec":float(row.cycle_onset_sec),"image_label":str(row.event_image_label),
                                           "event_dff":float(response[i]),"previous_image_dff":float(previous[i]),"matched_control_dff":matched_mean,
                                           "event_minus_previous_dff":float(response[i]-previous[i]),"event_minus_matched_dff":float(response[i]-matched_mean) if np.isfinite(matched_mean) else np.nan,
                                           "matched_control_onsets_sec":control_onsets})
                event_curve_rows.append({**meta,"event_type":event_type,"time":t,"mean_dff":np.nanmean(aligned,axis=0),"n_trials":len(aligned)})

image_trial_df=pd.concat(image_trial_rows,ignore_index=True) if image_trial_rows else pd.DataFrame()
image_curve_df=pd.DataFrame(image_curve_rows)
event_dff_df=pd.DataFrame(event_dff_rows)
event_curve_df=pd.DataFrame(event_curve_rows)

print(f"Image trials: {len(image_trial_df):,} · event trials: {len(event_dff_df):,}")
display(event_dff_df.head())


## Add spike counts to the exact same image/change/omission events


In [ ]:
# Ordinary image spike counts.
image_spike_rows=[]
if len(spikes):
    for keys,g in image_trial_df.groupby(["session_id","dmd","roi"],observed=True):
        sid,dmd,roi=keys; st=spikes[(spikes["session_id"].astype(str)==str(sid))&(spikes["dmd"].astype(int)==int(dmd))&(spikes["roi"].astype(int)==int(roi))]["spike_time_sec"].to_numpy(float)
        q=g.copy(); counts=spike_counts(st,q["onset_sec"].to_numpy(float),IMAGE_RESPONSE_S); q["image_spike_count"]=counts; q["image_spike_rate_hz"]=counts/(IMAGE_RESPONSE_S[1]-IMAGE_RESPONSE_S[0]); image_spike_rows.append(q)
image_trial_spike_df=pd.concat(image_spike_rows,ignore_index=True) if image_spike_rows else image_trial_df.copy()

# Change/omission event-vs-matched-control spike counts.
event_spike_rows=[]
if len(spikes):
    for keys,g in event_dff_df.groupby(["session_id","dmd","roi"],observed=True):
        sid,dmd,roi=keys; st=spikes[(spikes["session_id"].astype(str)==str(sid))&(spikes["dmd"].astype(int)==int(dmd))&(spikes["roi"].astype(int)==int(roi))]["spike_time_sec"].to_numpy(float)
        for row in g.itertuples(index=False):
            window=CHANGE_RESPONSE_S if row.event_type=="change" else OMISSION_RESPONSE_S; dur=window[1]-window[0]
            event_count=int(spike_counts(st,[row.onset_sec],window)[0]); ctrl=np.asarray(row.matched_control_onsets_sec,float)
            ctrl_counts=spike_counts(st,ctrl,window).astype(float) if len(ctrl) else np.array([],float); ctrl_mean=float(np.mean(ctrl_counts)) if len(ctrl_counts) else np.nan
            d=row._asdict(); d.update(event_spike_count=event_count,event_spike_rate_hz=event_count/dur,matched_control_spike_count=ctrl_mean,
                                     matched_control_spike_rate_hz=ctrl_mean/dur if np.isfinite(ctrl_mean) else np.nan,
                                     event_minus_matched_spike_count=event_count-ctrl_mean if np.isfinite(ctrl_mean) else np.nan,
                                     event_minus_matched_spike_rate_hz=(event_count-ctrl_mean)/dur if np.isfinite(ctrl_mean) else np.nan)
            event_spike_rows.append(d)
event_trial_df=pd.DataFrame(event_spike_rows) if event_spike_rows else event_dff_df.copy()

display(event_trial_df.head())


## Cell-by-cell response statistics

A dF/F change/omission response is called **clear** only when the cell has at least `MIN_EVENT_TRIALS`, the paired event-minus-control signed-rank test survives BH-FDR, and the bootstrap 95% CI on the mean paired effect excludes zero. There is deliberately no arbitrary minimum dF/F amplitude threshold. Spike responses are screened independently with the same paired logic.

Image identity is assessed with trial-wise FVE and a permutation test after requiring enough repeats per identity. This is a screen for reproducible identity dependence, not a claim that FVE is unbiased or that trials are perfectly independent.


In [ ]:
def summarize_paired(g,effect_col,prefix):
    x=pd.to_numeric(g[effect_col],errors="coerce").to_numpy(float); x=x[np.isfinite(x)]; lo,hi=bootstrap_mean_ci(x)
    return pd.Series({f"{prefix}_n":len(x),f"{prefix}_mean":np.mean(x) if len(x) else np.nan,f"{prefix}_median":np.median(x) if len(x) else np.nan,
                      f"{prefix}_ci_low":lo,f"{prefix}_ci_high":hi,f"{prefix}_p":signed_rank_p(x),
                      f"{prefix}_positive_fraction":np.mean(x>0) if len(x) else np.nan})

def summarize_identity(g,value_col,prefix):
    q=g[["image_label",value_col]].copy(); q[value_col]=pd.to_numeric(q[value_col],errors="coerce"); q=q.dropna()
    counts=q["image_label"].astype(str).value_counts(); keep=counts[counts>=MIN_TRIALS_PER_IMAGE].index.astype(str); q=q[q["image_label"].astype(str).isin(keep)]
    fve=identity_fve(q["image_label"],q[value_col]) if q["image_label"].nunique()>=MIN_IMAGES_FOR_IDENTITY else np.nan
    p=identity_permutation_p(q["image_label"],q[value_col]) if np.isfinite(fve) else np.nan
    return pd.Series({f"{prefix}_n":len(q),f"{prefix}_n_images":q["image_label"].nunique(),f"{prefix}_fve":fve,f"{prefix}_p":p})

GROUP=["subject_id","session_id","session_label","session_order","dmd","roi","cell_id","global_cell_id","manually_registered","depth_um","depth_group"]

image_dff_summary=(image_trial_df.groupby(GROUP,observed=True,dropna=False).apply(lambda g:summarize_identity(g,"image_delta_dff","image_dff"),include_groups=False).reset_index())
image_spike_summary=(image_trial_spike_df.groupby(GROUP,observed=True,dropna=False).apply(lambda g:summarize_identity(g,"image_spike_rate_hz","image_spike"),include_groups=False).reset_index()) if "image_spike_rate_hz" in image_trial_spike_df else pd.DataFrame(columns=GROUP)

pair_summaries={}
for event_type in ["change","omission"]:
    q=event_trial_df[event_trial_df["event_type"].astype(str).eq(event_type)]
    pair_summaries[(event_type,"dff")]=(q.groupby(GROUP,observed=True,dropna=False).apply(lambda g:summarize_paired(g,"event_minus_matched_dff",f"{event_type}_dff"),include_groups=False).reset_index())
    if "event_minus_matched_spike_rate_hz" in q:
        pair_summaries[(event_type,"spike")]=(q.groupby(GROUP,observed=True,dropna=False).apply(lambda g:summarize_paired(g,"event_minus_matched_spike_rate_hz",f"{event_type}_spike"),include_groups=False).reset_index())

response_table=image_dff_summary.copy()
for table in [image_spike_summary,pair_summaries.get(("change","dff")),pair_summaries.get(("change","spike")),pair_summaries.get(("omission","dff")),pair_summaries.get(("omission","spike"))]:
    if table is not None and len(table): response_table=response_table.merge(table,on=GROUP,how="outer")

# FDR is applied separately to each biological readout.
for prefix in ["image_dff","image_spike","change_dff","change_spike","omission_dff","omission_spike"]:
    pcol=f"{prefix}_p"
    if pcol in response_table:
        response_table[f"{prefix}_q"]=bh_fdr(response_table[pcol].to_numpy(float))

response_table["clear_image_identity_dff"]=(response_table.get("image_dff_q",np.nan)<FDR_ALPHA)&(response_table.get("image_dff_n_images",0)>=MIN_IMAGES_FOR_IDENTITY)
response_table["clear_image_identity_spike"]=(response_table.get("image_spike_q",np.nan)<FDR_ALPHA)&(response_table.get("image_spike_n_images",0)>=MIN_IMAGES_FOR_IDENTITY)
for event_type in ["change","omission"]:
    for modality in ["dff","spike"]:
        prefix=f"{event_type}_{modality}"; q=response_table.get(f"{prefix}_q",pd.Series(np.nan,index=response_table.index)); n=response_table.get(f"{prefix}_n",pd.Series(0,index=response_table.index))
        lo=response_table.get(f"{prefix}_ci_low",pd.Series(np.nan,index=response_table.index)); hi=response_table.get(f"{prefix}_ci_high",pd.Series(np.nan,index=response_table.index))
        response_table[f"clear_{event_type}_{modality}"]=(n>=MIN_EVENT_TRIALS)&(q<FDR_ALPHA)&((lo>0)|(hi<0))
        mean=response_table.get(f"{prefix}_mean",pd.Series(np.nan,index=response_table.index))
        response_table[f"{event_type}_{modality}_direction"]=np.where(mean>0,"activated",np.where(mean<0,"suppressed","none"))

valid_global=response_table["global_cell_id"].astype(str).replace("nan","").ne("")
response_table["dataset_cell_key"]=np.where(valid_global,
    response_table["subject_id"].astype(str)+":G"+response_table["global_cell_id"].astype(str),
    response_table["subject_id"].astype(str)+":"+response_table["session_id"].astype(str)+f":DMD"+response_table["dmd"].astype(str)+":ROI"+response_table["roi"].astype(str))

display(response_table.sort_values(["subject_id","session_order","dmd","roi"]).head(12))


In [ ]:
# Counts at both session × neuron and unique-cell levels.
signals=[("Image identity","image_identity"),("Change","change"),("Omission","omission")]
rows=[]
for label,sig in signals:
    dff_col="clear_image_identity_dff" if sig=="image_identity" else f"clear_{sig}_dff"
    spike_col="clear_image_identity_spike" if sig=="image_identity" else f"clear_{sig}_spike"
    tested_dff=response_table[dff_col].notna(); tested_spike=response_table[spike_col].notna()
    rows.append(dict(signal=label,roi_observations=len(response_table),clear_dff_observations=int(response_table[dff_col].fillna(False).sum()),clear_spike_observations=int(response_table[spike_col].fillna(False).sum()),
                     unique_cells=int(response_table["dataset_cell_key"].nunique()),unique_cells_clear_dff=int(response_table.groupby("dataset_cell_key")[dff_col].any().sum()),unique_cells_clear_spike=int(response_table.groupby("dataset_cell_key")[spike_col].any().sum())))
count_table=pd.DataFrame(rows)

unique_cell_summary=(response_table.groupby(["subject_id","dataset_cell_key"],observed=True).agg(
    n_sessions=("session_id","nunique"),tracked=("manually_registered","max"),depth_um=("depth_um","median"),
    image_identity_dff_ever=("clear_image_identity_dff","max"),change_dff_ever=("clear_change_dff","max"),omission_dff_ever=("clear_omission_dff","max"),
    image_identity_spike_ever=("clear_image_identity_spike","max"),change_spike_ever=("clear_change_spike","max"),omission_spike_ever=("clear_omission_spike","max"),
).reset_index())
unique_cell_summary["depth_group"]=unique_cell_summary["depth_um"].map(depth_group_from_um)

display(count_table)
display(unique_cell_summary.sort_values(["tracked","n_sessions"],ascending=False).head(20))

response_table.to_csv(CACHE_DIR/"asap8_cell_session_response_table.csv",index=False)
unique_cell_summary.to_csv(CACHE_DIR/"asap8_unique_cell_response_summary.csv",index=False)


In [ ]:
# Session-specific responder counts: the most direct answer to "how many cells?" without conflating repeat observations.
summary_by_session=[]
for label in SESSION_ORDER:
    q=response_table[response_table["session_label"].astype(str).eq(label)]
    if q.empty: continue
    summary_by_session.append(dict(session=label,n_cells=len(q),image_identity=int(q["clear_image_identity_dff"].sum()),change=int(q["clear_change_dff"].sum()),omission=int(q["clear_omission_dff"].sum()),change_spike=int(q["clear_change_spike"].sum()),omission_spike=int(q["clear_omission_spike"].sum())))
summary_by_session=pd.DataFrame(summary_by_session)
display(summary_by_session)

fig,axs=plt.subplots(1,2,figsize=(8.5,3.4),sharey=False)
x=np.arange(3); labels=["Image identity","Change","Omission"]
axs[0].bar(x,[count_table.loc[i,"clear_dff_observations"] for i in range(3)],width=.65)
axs[0].set(xticks=x,xticklabels=labels,ylabel="Session × neuron observations",title="Clear dF/F signals")
axs[1].bar(x,[count_table.loc[i,"unique_cells_clear_dff"] for i in range(3)],width=.65)
axs[1].set(xticks=x,xticklabels=labels,ylabel="Unique cells",title="Cells with ≥1 clear session")
for ax in axs: finish_axis(ax); ax.tick_params(axis="x",rotation=25)
fig.tight_layout(); plt.show()


## Individual-cell membrane-potential atlas


In [ ]:
def plot_cell_trace_atlas(event_type,only_clear=False,max_cells=36,ncols=6):
    q=event_curve_df[event_curve_df["event_type"].astype(str).eq(event_type)].copy()
    metric=f"{event_type}_dff_mean"; clear=f"clear_{event_type}_dff"
    q=q.merge(response_table[["session_id","dmd","roi",metric,clear]],on=["session_id","dmd","roi"],how="left")
    if only_clear: q=q[q[clear].fillna(False)]
    q=q.sort_values(metric,key=lambda x:x.abs(),ascending=False).head(int(max_cells))
    if q.empty: print(f"No {event_type} curves to plot"); return
    ncols=min(ncols,len(q)); nrows=int(np.ceil(len(q)/ncols)); fig,axs=plt.subplots(nrows,ncols,figsize=(2.25*ncols,1.75*nrows),sharex=True); axs=np.atleast_1d(axs).ravel()
    window=CHANGE_RESPONSE_S if event_type=="change" else OMISSION_RESPONSE_S
    for ax,row in zip(axs,q.itertuples(index=False)):
        ax.plot(row.time,row.mean_dff,color=DEPTH_COLORS[str(row.depth_group)],lw=1.0)
        ax.axvspan(*window,color=".85",alpha=.5,lw=0); ax.axvline(0,color=".4",ls="--",lw=.7)
        ax.set_title(f"M{row.subject_id} {row.session_label} D{row.dmd}R{row.roi}\nΔ={getattr(row,metric):+.3g}",fontsize=8); finish_axis(ax)
    for ax in axs[len(q):]: ax.axis("off")
    fig.suptitle(f"{event_type.capitalize()}-aligned mean dF/F by individual cell"+(" · clear responders" if only_clear else ""),y=1.01)
    fig.supxlabel(f"Time from {event_type} (s)"); fig.supylabel("dF/F"); fig.tight_layout(); plt.show()

plot_cell_trace_atlas("change",only_clear=False)
plot_cell_trace_atlas("omission",only_clear=False)


## Trial-by-trial evidence panels for automatically selected clear responders


In [ ]:
def _select_response_row(signal,rank=0):
    if signal=="image_identity": q=response_table[response_table["clear_image_identity_dff"].fillna(False)].sort_values("image_dff_fve",ascending=False)
    else: q=response_table[response_table[f"clear_{signal}_dff"].fillna(False)].assign(_abs=lambda x:x[f"{signal}_dff_mean"].abs()).sort_values("_abs",ascending=False)
    if q.empty: raise ValueError(f"No clear {signal} dF/F responders under current criteria")
    return q.iloc[int(rank)%len(q)]

def _load_event_trials(row,event_type):
    s=sessions[sessions["session_id"].astype(str).eq(str(row.session_id))].iloc[0]
    with h5py.File(s["single_trial_h5"],"r") as h5:
        g=h5[f"DMD{int(row.dmd)}"]; axis=h5_roi_axis(g,int(row.dmd),int(row.roi)); traces=np.asarray(g[event_type]["traces"][:,axis,:],float); t=reconcile_timebase(h5[f"timebase_sec/{event_type}"][:],traces.shape[-1]); idx=indexed_event_table(str(row.session_id),int(row.dmd),event_type)
        offsets=pd.to_numeric(idx["cycle_onset_sec"],errors="coerce").to_numpy()-pd.to_numeric(idx["stored_onset_sec"],errors="coerce").to_numpy(); aligned=align_trials_to_true_onset(traces,t,offsets)
    return t,aligned,idx

def plot_event_cell_evidence(row,event_type,max_trials=60):
    t,traces,idx=_load_event_trials(row,event_type); q=event_trial_df[(event_trial_df["session_id"].astype(str)==str(row.session_id))&(event_trial_df["dmd"].astype(int)==int(row.dmd))&(event_trial_df["roi"].astype(int)==int(row.roi))&(event_trial_df["event_type"].astype(str)==event_type)].sort_values("trial_index")
    take=np.arange(len(traces));
    if len(take)>max_trials: take=np.unique(np.linspace(0,len(traces)-1,max_trials).round().astype(int))
    window=CHANGE_RESPONSE_S if event_type=="change" else OMISSION_RESPONSE_S; c=SIGNAL_COLORS[event_type]
    fig,axs=plt.subplots(2,2,figsize=(9.2,6.6))
    for y in traces[take]: axs[0,0].plot(t,y,color=c,lw=.45,alpha=.16)
    axs[0,0].plot(t,np.nanmean(traces[take],axis=0),color="black",lw=1.5); axs[0,0].axvspan(*window,color=".85",alpha=.55,lw=0); axs[0,0].axvline(0,color=".35",ls="--",lw=.8); axs[0,0].set(title="Raw dF/F trials + mean",xlabel=f"Time from {event_type} (s)",ylabel="dF/F"); finish_axis(axs[0,0])
    finite=traces[take][np.isfinite(traces[take])]; lim=np.nanpercentile(finite,[2,98]) if len(finite) else (-1,1); im=axs[0,1].imshow(traces[take],aspect="auto",interpolation="nearest",extent=[t[0],t[-1],len(take)-.5,-.5],vmin=lim[0],vmax=lim[1],cmap="viridis"); axs[0,1].axvline(0,color="white",ls="--",lw=.8); axs[0,1].set(title="Trial × time dF/F",xlabel="Time (s)",ylabel="Trial"); plt.colorbar(im,ax=axs[0,1],label="dF/F")
    good=q[["matched_control_dff","event_dff"]].dropna(); x=np.arange(len(good)); axs[1,0].plot(np.c_[np.zeros(len(good)),np.ones(len(good))].T,np.c_[good["matched_control_dff"],good["event_dff"]].T,color=".75",lw=.6); axs[1,0].scatter(np.zeros(len(good)),good["matched_control_dff"],s=12,color=".45"); axs[1,0].scatter(np.ones(len(good)),good["event_dff"],s=12,color=c); axs[1,0].set(xticks=[0,1],xticklabels=["Matched\nordinary","Event"],ylabel="Window dF/F",title=f"Paired dF/F · mean Δ={q['event_minus_matched_dff'].mean():+.3g}"); finish_axis(axs[1,0])
    # Spike raster in the same trials.
    st=spikes[(spikes["session_id"].astype(str)==str(row.session_id))&(spikes["dmd"].astype(int)==int(row.dmd))&(spikes["roi"].astype(int)==int(row.roi))]["spike_time_sec"].to_numpy(float) if len(spikes) else np.array([])
    onsets=pd.to_numeric(idx["cycle_onset_sec"],errors="coerce").to_numpy(float); raster_window=(max(t[0],-1.0),min(t[-1],1.25))
    for j,onset in enumerate(onsets[take]):
        rel=st[(st>=onset+raster_window[0])&(st<onset+raster_window[1])]-onset; axs[1,1].vlines(rel,j-.38,j+.38,color="black",lw=.65)
    axs[1,1].axvspan(*window,color=".85",alpha=.55,lw=0); axs[1,1].axvline(0,color=".35",ls="--",lw=.8); axs[1,1].set(xlim=raster_window,ylim=(len(take)-.5,-.5),xlabel=f"Time from {event_type} (s)",ylabel="Trial",title="Detected spikes on the same events"); finish_axis(axs[1,1])
    fig.suptitle(f"M{row.subject_id} · {row.session_label} · DMD{int(row.dmd)} ROI{int(row.roi)} · {row.depth_um:.0f} µm",y=.995); fig.tight_layout(); plt.show()

def plot_image_identity_evidence(row,max_trials=100):
    q=image_trial_spike_df[(image_trial_spike_df["session_id"].astype(str)==str(row.session_id))&(image_trial_spike_df["dmd"].astype(int)==int(row.dmd))&(image_trial_spike_df["roi"].astype(int)==int(row.roi))].copy()
    s=sessions[sessions["session_id"].astype(str).eq(str(row.session_id))].iloc[0]
    with h5py.File(s["single_trial_h5"],"r") as h5:
        g=h5[f"DMD{int(row.dmd)}"]; axis=h5_roi_axis(g,int(row.dmd),int(row.roi)); controls=load_image_control_table(h5,str(row.session_id),int(row.dmd),int(row.roi),axis); stored_t=np.asarray(h5["timebase_sec/image"][:],float); traces=[]; labels=[]
        for path,h in controls.groupby("dataset_path",sort=False):
            rows=h["trial_index"].astype(int).to_numpy(); ds=h5[str(path)]; t=reconcile_timebase(stored_t,int(ds.shape[-1])); arr=np.asarray(ds[rows,axis,:],float); traces.append(arr); labels.extend(h["event_image_label"].astype(str).tolist())
    traces=np.concatenate(traces,axis=0); labels=np.asarray(labels); order=np.argsort(labels); traces,labels=traces[order],labels[order]; take=np.arange(min(len(traces),max_trials)); uniq=pd.unique(labels); cmap=plt.cm.tab10(np.linspace(0,1,max(3,len(uniq)))); colors={lab:cmap[i] for i,lab in enumerate(uniq)}
    fig,axs=plt.subplots(2,2,figsize=(9.2,6.6))
    for lab in uniq:
        arr=traces[labels==lab]; axs[0,0].plot(t,np.nanmean(arr,axis=0),color=colors[lab],lw=1.2,label=Path(str(lab).replace('\\','/')).stem)
    axs[0,0].axvspan(*IMAGE_RESPONSE_S,color=".85",alpha=.5,lw=0); axs[0,0].axvline(0,color=".35",ls="--",lw=.8); axs[0,0].set(title="Mean dF/F by image identity",xlabel="Time from image onset (s)",ylabel="dF/F"); finish_axis(axs[0,0]); axs[0,0].legend(frameon=False,fontsize=7,ncol=2)
    finite=traces[take][np.isfinite(traces[take])]; lim=np.nanpercentile(finite,[2,98]) if len(finite) else (-1,1); im=axs[0,1].imshow(traces[take],aspect="auto",interpolation="nearest",extent=[t[0],t[-1],len(take)-.5,-.5],vmin=lim[0],vmax=lim[1],cmap="viridis"); axs[0,1].axvline(0,color="white",ls="--",lw=.8); axs[0,1].set(title="Ordinary image trials sorted by identity",xlabel="Time (s)",ylabel="Trial"); plt.colorbar(im,ax=axs[0,1],label="dF/F")
    sns.stripplot(data=q,x="image_label",y="image_delta_dff",ax=axs[1,0],size=3,alpha=.5); axs[1,0].tick_params(axis="x",rotation=45); axs[1,0].set(xlabel="Image",ylabel="0–0.25 s − baseline dF/F",title=f"Identity FVE={row.image_dff_fve:.3f}, q={row.image_dff_q:.3g}"); finish_axis(axs[1,0])
    if "image_spike_rate_hz" in q:
        sns.stripplot(data=q,x="image_label",y="image_spike_rate_hz",ax=axs[1,1],size=3,alpha=.5); axs[1,1].tick_params(axis="x",rotation=45); axs[1,1].set(xlabel="Image",ylabel="Spike rate (Hz)",title=f"Spike identity FVE={getattr(row,'image_spike_fve',np.nan):.3f}"); finish_axis(axs[1,1])
    else: axs[1,1].axis("off")
    fig.suptitle(f"M{row.subject_id} · {row.session_label} · DMD{int(row.dmd)} ROI{int(row.roi)} · {row.depth_um:.0f} µm",y=.995); fig.tight_layout(); plt.show()


In [ ]:
N_EXAMPLES_PER_SIGNAL = 2
for signal in ["image_identity","change","omission"]:
    print(f"\n===== {signal.upper()} =====")
    try:
        for rank in range(N_EXAMPLES_PER_SIGNAL):
            row=_select_response_row(signal,rank)
            if signal=="image_identity": plot_image_identity_evidence(row)
            else: plot_event_cell_evidence(row,signal)
    except ValueError as exc:
        print(exc)


## Optional: export a trial-level evidence figure for every clear responder


In [ ]:
EXPORT_ALL_CLEAR_CELL_REPORTS = False
if EXPORT_ALL_CLEAR_CELL_REPORTS:
    for signal in ["image_identity","change","omission"]:
        clear_col="clear_image_identity_dff" if signal=="image_identity" else f"clear_{signal}_dff"
        for row in response_table[response_table[clear_col].fillna(False)].itertuples(index=False):
            if signal=="image_identity": plot_image_identity_evidence(row,max_trials=150)
            else: plot_event_cell_evidence(row,signal,max_trials=100)


## Interpretation checklist

For the conversation with Kaspar, start with `count_table` and `summary_by_session`, then inspect the change/omission atlases before opening the strongest trial-level examples. A convincing cell should have a visible event-locked dF/F difference across many trials, a paired event-vs-matched-control distribution that is not driven by one or two outliers, and—where present—corroborating changes in spike timing/rate. Cells with a clear subthreshold dF/F signal but little spiking are still biologically informative and are intentionally retained as a separate category rather than discarded.
